# GLP-1 Prescriber Targeting (Medicare Part D)

**Question:** which prescribers should a GLP-1 diabetes brand's sales team visit first?

**Data:** [Medicare Part D Prescribers – by Provider and Drug](https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug) from CMS. Download the latest year's CSV and save it as `data/partd_prescriber_drug.csv` (see `data/README.md`).

In [1]:
import duckdb
import pandas as pd
import sys
print("Python:",sys.version)
print("DuckDB:",duckdb.__version__)
print("Pandas:",pd.__version__)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
DuckDB: 2.0.0
Pandas: 3.0.6


## 1. Load the data with SQL (DuckDB)

Loading the full file takes a few minutes. Note the row count: it goes in the README and on the resume.

In [2]:
import duckdb   # DuckDB: a small database engine that runs SQL directly on files

# Create (or open) a database file on disk. Tables you create are saved in it,
# so you don't have to reload the big CSV every time you restart the notebook.
con = duckdb.connect('partd.duckdb')

# Load the whole CMS CSV into a table called rx.
# read_csv_auto works out the column names and types by itself;
# sample_size=-1 makes it scan every row, so no column ends up with the wrong type.
con.execute('''
    CREATE OR REPLACE TABLE rx AS
    SELECT * FROM read_csv_auto('data/partd_prescriber_drug.csv', sample_size=-1)
''')

# List the columns and their types: check the names before writing more SQL.
# print() is needed because a notebook only displays the last result of a cell.
print(con.sql('DESCRIBE rx').df())

# Count rows and unique prescribers (NPI = National Provider Identifier, one per prescriber).
# .df() turns the SQL result into a pandas DataFrame, which displays as a table.
con.sql('SELECT COUNT(*) AS rows, COUNT(DISTINCT Prscrbr_NPI) AS prescribers FROM rx').df()

              column_name column_type null   key default extra
0             Prscrbr_NPI      BIGINT  YES  None    None  None
1   Prscrbr_Last_Org_Name     VARCHAR  YES  None    None  None
2      Prscrbr_First_Name     VARCHAR  YES  None    None  None
3            Prscrbr_City     VARCHAR  YES  None    None  None
4    Prscrbr_State_Abrvtn     VARCHAR  YES  None    None  None
5      Prscrbr_State_FIPS     VARCHAR  YES  None    None  None
6            Prscrbr_Type     VARCHAR  YES  None    None  None
7        Prscrbr_Type_Src     VARCHAR  YES  None    None  None
8               Brnd_Name     VARCHAR  YES  None    None  None
9               Gnrc_Name     VARCHAR  YES  None    None  None
10               Tot_Clms      BIGINT  YES  None    None  None
11        Tot_30day_Fills      DOUBLE  YES  None    None  None
12          Tot_Day_Suply      BIGINT  YES  None    None  None
13           Tot_Drug_Cst      DOUBLE  YES  None    None  None
14              Tot_Benes      BIGINT  YES  None    Non

,rows,prescribers
0,26794878,1104162


## 2. Pull out the GLP-1 drugs

First check how CMS spells the molecule names, then build the two working tables.

In [3]:
# Before filtering, see exactly how CMS spells the GLP-1 drugs.
# lower() makes the match ignore capitals; regexp_matches() looks for any of the names,
# where | means "or". GROUP BY ALL groups by every column that isn't added up.
con.sql('''
    SELECT Gnrc_Name, Brnd_Name, SUM(Tot_Clms) AS claims
    FROM rx
    WHERE regexp_matches(lower(Gnrc_Name), 'semaglutide|tirzepatide|dulaglutide|liraglutide|exenatide')
    GROUP BY ALL
    ORDER BY claims DESC
''').df()

,Gnrc_Name,Brnd_Name,claims
0,Semaglutide,Ozempic,6416007.0
1,Dulaglutide,Trulicity,4851616.0
2,Tirzepatide,Mounjaro,1521598.0
3,Semaglutide,Rybelsus,755052.0
4,Liraglutide,Victoza 3-Pak,309275.0
5,Liraglutide,Victoza 2-Pak,116472.0
6,Exenatide Microspheres,Bydureon Bcise,109771.0
7,Insulin Degludec/Liraglutide,Xultophy 100-3.6,25493.0
8,Exenatide,Byetta,2699.0
9,Semaglutide,Wegovy,26.0


In [4]:
# Table 1, glp1: one row per prescriber per GLP-1 brand, with short, readable column names.
# concat_ws joins last and first name with ', ' and skips the first name when it is empty
# (organisations have no first name).
con.execute('''
    CREATE OR REPLACE TABLE glp1 AS
    SELECT Prscrbr_NPI AS npi,
           concat_ws(', ', Prscrbr_Last_Org_Name, Prscrbr_First_Name) AS prescriber,
           Prscrbr_State_Abrvtn AS state,
           Prscrbr_Type AS specialty,
           Gnrc_Name AS molecule,
           Tot_Clms AS claims,
           Tot_Drug_Cst AS drug_cost
    FROM rx
    WHERE regexp_matches(lower(Gnrc_Name), 'semaglutide|tirzepatide|dulaglutide|liraglutide|exenatide')
''')

# Table 2, prescribers: add up each prescriber's rows, so there is exactly one row per prescriber.
# total_claims = all GLP-1 prescriptions they wrote; molecules_used = how many different GLP-1s.
con.execute('''
    CREATE OR REPLACE TABLE prescribers AS
    SELECT npi, prescriber, state, specialty,
           SUM(claims) AS total_claims,
           SUM(drug_cost) AS total_cost,
           COUNT(DISTINCT molecule) AS molecules_used
    FROM glp1
    GROUP BY ALL
''')

## 3. Decile the prescribers by volume

Decile 10 is the small group writing the first 10% of all claims; decile 1 is the long tail writing the last 10%.

In [5]:
# Volume-based deciles, in two parts:
# 1. ranked: sort prescribers from most to fewest claims and keep a running total (cum_claims).
#    SUM(...) OVER (ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) adds up the
#    claims of this prescriber and everyone ranked above them. all_claims is the grand total.
#    npi breaks ties, so prescribers with equal claims always sort the same way.
# 2. decile: (cum_claims - total_claims) / all_claims is the share of claims written by everyone
#    ranked above this prescriber. 0-10% gives decile 10, 10-20% gives decile 9 ... 90-100% gives 1.
#    LEAST(..., 9) is a safety guard that keeps the result between 1 and 10.
con.execute('''
    CREATE OR REPLACE TABLE deciles AS
    WITH ranked AS (
        SELECT *,
               SUM(total_claims) OVER (ORDER BY total_claims DESC, npi
                                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_claims,
               SUM(total_claims) OVER () AS all_claims
        FROM prescribers
    )
    SELECT *,
           10 - LEAST(CAST(FLOOR(10.0 * (cum_claims - total_claims) / all_claims) AS INTEGER), 9) AS decile
    FROM ranked
''')

# Summary per decile. SUM(...) OVER () is the grand total across all deciles, so each row's
# share can be calculated in the same query. pct_of_claims should be about 10 in every row;
# pct_of_prescribers is the interesting column.
con.sql('''
    SELECT decile,
           COUNT(*) AS prescribers,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_prescribers,
           ROUND(100.0 * SUM(total_claims) / SUM(SUM(total_claims)) OVER (), 1) AS pct_of_claims
    FROM deciles
    GROUP BY decile
    ORDER BY decile DESC
''').df()

,decile,prescribers,pct_of_prescribers,pct_of_claims
0,10,1545,0.9,10.0
1,9,2941,1.7,10.0
2,8,4515,2.6,10.0
3,7,6384,3.7,10.0
4,6,8558,5.0,10.0
5,5,11195,6.5,10.0
6,4,14783,8.6,10.0
7,3,20223,11.8,10.0
8,2,30652,17.8,10.0
9,1,71027,41.3,10.0


**What I found:** _add up pct_of_prescribers for deciles 6–10: that share of prescribers writes half of all GLP-1 claims._

## 4. Find the high-value segments

In [6]:
# One row per specialty:
#   claims                 = total GLP-1 claims written by that specialty
#   claims_per_prescriber  = how much a typical prescriber in that specialty writes
#   top_decile_prescribers = how many of them sit in deciles 8-10, the high-value targets
# CASE WHEN ... THEN 1 ELSE 0 END turns "is in deciles 8-10" into 1 or 0, and SUM counts the 1s.
con.sql('''
    SELECT specialty,
           COUNT(*) AS prescribers,
           SUM(total_claims) AS claims,
           ROUND(AVG(total_claims), 1) AS claims_per_prescriber,
           SUM(CASE WHEN decile >= 8 THEN 1 ELSE 0 END) AS top_decile_prescribers
    FROM deciles
    GROUP BY specialty
    ORDER BY claims DESC
    LIMIT 15
''').df()

,specialty,prescribers,claims,claims_per_prescriber,top_decile_prescribers
0,Family Practice,56362,4064455.0,72.1,1553.0
1,Internal Medicine,40727,3567036.0,87.6,2116.0
2,Nurse Practitioner,42151,2810795.0,66.7,1521.0
3,Endocrinology,5725,2003908.0,350.0,2944.0
4,Physician Assistant,14362,946944.0,65.9,498.0
5,General Practice,2093,140834.0,67.3,74.0
6,Cardiology,1539,103895.0,67.5,64.0
7,Pharmacist,860,69050.0,80.3,35.0
8,Nephrology,802,56801.0,70.8,43.0
9,Geriatric Medicine,704,43945.0,62.4,17.0


In [7]:
# The same view by state. print() is needed because a notebook only displays a cell's last result.
print(con.sql('''
    SELECT state,
           COUNT(*) AS prescribers,
           SUM(total_claims) AS claims,
           ROUND(AVG(total_claims), 1) AS claims_per_prescriber,
           SUM(CASE WHEN decile >= 8 THEN 1 ELSE 0 END) AS top_decile_prescribers
    FROM deciles
    GROUP BY state
    ORDER BY claims DESC
''').df())

# Molecule share within the 5 biggest specialties:
# - top_specialties is a named sub-query (a CTE) that finds the 5 specialties with the most claims
# - the main query adds up claims for each specialty and molecule
# - SUM(SUM(claims)) OVER (PARTITION BY specialty) is that specialty's total, so pct_of_specialty
#   is each molecule's share within its own specialty
con.sql('''
    WITH top_specialties AS (
        SELECT specialty FROM prescribers
        GROUP BY specialty
        ORDER BY SUM(total_claims) DESC
        LIMIT 5
    )
    SELECT specialty, molecule,
           SUM(claims) AS claims,
           ROUND(100.0 * SUM(claims) / SUM(SUM(claims)) OVER (PARTITION BY specialty), 1) AS pct_of_specialty
    FROM glp1
    WHERE specialty IN (SELECT specialty FROM top_specialties)
    GROUP BY specialty, molecule
    ORDER BY specialty, claims DESC
''').df()

   state  prescribers     claims  claims_per_prescriber  \
0     TX        12884  1185513.0                   92.0   
1     CA        15290  1154671.0                   75.5   
2     NY        10928   995822.0                   91.1   
3     FL        11134   879134.0                   79.0   
4     PA         7868   686501.0                   87.3   
5     NC         6550   580051.0                   88.6   
6     OH         6874   558643.0                   81.3   
7     MI         6238   497627.0                   79.8   
8     GA         4799   488364.0                  101.8   
9     IL         5666   431174.0                   76.1   
10    TN         4277   393292.0                   92.0   
11    IN         4007   372817.0                   93.0   
12    AL         2807   364074.0                  129.7   
13    MA         4500   360975.0                   80.2   
14    LA         2939   351854.0                  119.7   
15    MO         3547   328256.0                   92.5 

,specialty,molecule,claims,pct_of_specialty
0,Endocrinology,Semaglutide,977954.0,48.8
1,Endocrinology,Dulaglutide,700066.0,34.9
2,Endocrinology,Tirzepatide,234036.0,11.7
3,Endocrinology,Liraglutide,67925.0,3.4
4,Endocrinology,Exenatide Microspheres,16631.0,0.8
5,Endocrinology,Insulin Degludec/Liraglutide,7075.0,0.4
6,Endocrinology,Exenatide,221.0,0.0
7,Family Practice,Semaglutide,2057694.0,50.6
8,Family Practice,Dulaglutide,1419979.0,34.9
9,Family Practice,Tirzepatide,409800.0,10.1


**What I found:** _which specialty writes the most claims in total, which writes the most per prescriber, and which molecule leads?_

## 5. Export for Power BI

Then build the dashboard in Power BI Desktop (Get data → Text/CSV) and save it as `dashboard.pbix`. The DAX measures for the KPI cards are in `powerbi/measures.dax`.

In [8]:
# Save the two tables as CSV files for Power BI.
# EXCLUDE drops the two helper columns that were only needed to calculate the deciles.
# (HEADER) writes the column names as the first line of each file.
con.execute("COPY (SELECT * EXCLUDE (cum_claims, all_claims) FROM deciles) TO 'data/glp1_prescribers.csv' (HEADER)")
con.execute("COPY glp1 TO 'data/glp1_prescriber_molecule.csv' (HEADER)")